# OlmoEarth for Limpopo dry-season irrigated-area mapping

**Ai2 OlmoEarth workshop — IWMI use case.** Fine-tune the OlmoEarth foundation model to map
**monthly dry-season (May–Sep) irrigated areas at 10 m** in the Limpopo River Basin from
**Sentinel-1 + Sentinel-2**.

This notebook mirrors the structure of the AWF Kenya `OlmoEarthTutorial`, but is wired to our
own labels and pipeline. It is the deep-learning successor to our published Random Forest system
([Kiala et al. 2026](https://doi.org/10.1007/s43832-026-00360-z); RF baseline OA 0.80 / κ 0.60).

| | AWF tutorial (template) | **This use case (Limpopo)** |
|---|---|---|
| Task | LULC classification, 9 classes | **Irrigation, binary** (0 non-irrigated, 1 irrigated) |
| Labels | 1,460 points, `oe_labels.category` | **190 field points + ODC-distilled weak labels** |
| Sensors | Sentinel-2 only | **Sentinel-1 (VV/VH) + Sentinel-2 (12 bands)** |
| Timesteps | 12 monthly (T=12) | **single dry-season month (T=1)**, per label |
| Window | 63×63 @ 10 m | 32×32 @ 10 m (center-token label) |
| Model | OlmoEarth-Nano, frozen→fine-tune | **OlmoEarth-LARGE**, ODC-pretrain → field fine-tune |
| Data source | Planetary Computer | Planetary Computer (same) |
| Best result | 82% (Nano/S2) | **OA 0.874 / κ 0.748** (spatial CV, 190 pts) |

> **Runs in Google Colab** (GPU runtime — *Runtime → Change runtime type → GPU*; T4 works, A100
> is faster for the LARGE fine-tune). It is self-contained: it installs dependencies, ingests two
> small label GeoJSONs (upload or Google Drive), and pulls imagery live from Planetary Computer —
> no local files or Earth Engine needed. It also runs unchanged in the local `olmoearth` env.

## 0. How this maps to the workshop's *labeled-data requirements*

The workshop asks: *is the dataset sufficient, quality, representative — what's missing?* Our answer:

- **Format.** Point labels in GeoJSON with a nested `properties.oe_labels.category` int code — the
  exact schema the OlmoEarth tooling expects (Section 1 builds this from our data).
- **Sufficiency.** 190 field points is *below* the tutorial's 1,460, so we augment with **in-basin,
  multi-month ODC-distilled weak labels** (~hundreds/class/month) and use a **curriculum**:
  pretrain on ODC weak labels → fine-tune on the 190 ground-truth points. Field points remain the
  only validation set (no leakage from weak labels).
- **Representativeness.** Labels span the dry season (Jun–Sep) and the basin; validation uses
  **spatial** cross-validation (blocks) to avoid optimistic leakage.
- **What's missing / next.** Ground truth outside South Africa (Zimbabwe, Botswana, Mozambique).

## 1. Setup — install dependencies & check GPU

In [ ]:
# Install once per Colab session (~2-3 min). Safe to re-run.
%pip -q install olmoearth_pretrain scikit-learn einops huggingface_hub planetary-computer pystac-client odc-stac rasterio matplotlib folium

In [ ]:
import os, sys, json, warnings, numpy as np, torch, torch.nn as nn
warnings.filterwarnings('ignore')
from pathlib import Path
from einops import rearrange

IN_COLAB = 'google.colab' in sys.modules
DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
WORK = Path('/content') if IN_COLAB else Path('.').resolve()
WORK.mkdir(exist_ok=True, parents=True)
print('in Colab :', IN_COLAB)
print('device   :', DEV, '(', torch.cuda.get_device_name(0) if DEV.type=='cuda' else 'CPU', ')')
if DEV.type != 'cuda':
    print('WARNING: no GPU. In Colab: Runtime -> Change runtime type -> GPU (T4).')

## 2. Labeled data — build the OlmoEarth label GeoJSON

**OlmoEarth label schema** (per [docs: labeled-data-requirements](https://docs.olmoearth.allenai.org/model-fine-tuning/#labeled-data-requirements)):
every label needs **Location** (lat/lon, WGS84) + **Time** (a date or date range) + a **Target**
that is *Category*, *Number*, or *Boolean*. Accepted files: **CSV** (points) or **GeoJSON**
(points *or* polygons). Irrigation is a **Boolean** target (`irrigated`), and the task is
**per-pixel classification (segmentation)** — a supported task type.

We combine two sources into one `FeatureCollection` of **Point** features, each carrying the
Boolean `irrigated` target, an int `oe_labels.category` (for the rslearn/tutorial path), and a
`date` (mid-month — irrigation is a *monthly* signal):

1. **Field points** — `limpopo/data/reference/field_points.geojson` (190 ground-truthed points).
2. **ODC-distilled weak labels** — `train_odc_2024dry_hiconf.geojson` (high-confidence points
   sampled from the IWMI `irrigated_areas_limpopo` product where `map` & `filtered` agree; built
   by `scripts/23_extract_odc_samples.py`). These provide the **negative (non-irrigated) examples**
   the docs require, balanced per class per month.

### 2a. Get the label files from the workshop GitHub repo

The two label GeoJSONs live in this repo under `data/`. In Colab we just **clone the repo** — no
manual upload. (Running locally in the `olmoearth` env, it uses `limpopo/data/reference/` instead.)

> If the repo is **private**, put a GitHub token in `REPO_URL` (see the commented line).

In [ ]:
import subprocess
REPO_URL = 'https://github.com/ZoloKiala/OlmoEarth_workshop.git'
# private repo? use:  REPO_URL = 'https://<GITHUB_TOKEN>@github.com/ZoloKiala/OlmoEarth_workshop.git'
REPO_DIR = WORK / 'OlmoEarth_workshop'
FIELD_NAME = 'field_points.geojson'
ODC_NAME   = 'train_odc_2024dry_hiconf.geojson'

if IN_COLAB:
    if not REPO_DIR.exists():
        print('cloning', REPO_URL, '…')
        subprocess.run(['git', 'clone', '--depth', '1', '-q', REPO_URL, str(REPO_DIR)], check=True)
    DATA = REPO_DIR / 'data'
else:
    DATA = Path('limpopo/data/reference')   # local olmoearth env

FIELD, ODC = DATA / FIELD_NAME, DATA / ODC_NAME
assert FIELD.exists(), f'not found: {FIELD}  (is the repo public? else add a token to REPO_URL)'
assert ODC.exists(),   f'not found: {ODC}'
print('field labels:', FIELD)
print('odc labels  :', ODC)

In [ ]:
# --- read + normalize both label sets into the OlmoEarth schema ---
def _load_features(path):
    with open(path) as f:
        return json.load(f).get('features', [])

def _to_oe(feat, category, year, month, src):
    lon, lat = feat['geometry']['coordinates'][:2]
    # OlmoEarth requires Location + Time + Target. Target is Boolean here (irrigated),
    # also kept as an int `category` for the rslearn/tutorial path. Time = mid-month date.
    date = f'{int(year)}-{int(month):02d}-15'
    return {'type': 'Feature',
            'geometry': {'type': 'Point', 'coordinates': [lon, lat]},
            'properties': {'irrigated': bool(int(category)),        # Boolean target (docs schema)
                           'oe_labels': {'category': int(category)},# int label (tutorial/rslearn)
                           'date': date, 'year': int(year), 'month': int(month), 'src': src}}

# FIELD / ODC paths come from cell 2a. Field label property may vary — adapt _field_label if needed.
def _field_label(p):
    for k in ('irrigated', 'label', 'class', 'category', 'irr'):
        if k in p: return int(p[k])
    raise KeyError(f'no label property in {list(p)}')

oe_feats = []
for ft in _load_features(FIELD):
    p = ft['properties']
    oe_feats.append(_to_oe(ft, _field_label(p), p.get('year', 2024), p.get('month', 9), 'field'))
n_field = len(oe_feats)
for ft in _load_features(ODC):
    p = ft['properties']
    oe_feats.append(_to_oe(ft, p['irrigated'], p.get('year', 2024), p.get('month', 9), 'odc'))
n_odc = len(oe_feats) - n_field

OE_LABELS = WORK / 'oe_labels_irrigation.geojson'
with open(OE_LABELS, 'w') as f:
    json.dump({'type': 'FeatureCollection', 'features': oe_feats}, f)
print(f'field points : {n_field}')
print(f'ODC weak     : {n_odc}')
print('wrote        :', OE_LABELS)

## 3. Extract S1 + S2 patches (live from Planetary Computer)

These helpers (inlined from our `lib_oe.py`) pull a **single-month** S2 median + S1 mean per point
and center-crop a 32×32 @ 10 m window. **Sentinel-1 is optional per point** (masked when the basin
isn't covered that month) — how we handle S1's incomplete monthly coverage. No Earth Engine, no local
files: everything comes from the public Planetary Computer STAC.

In [ ]:
import planetary_computer as pc, pystac_client
from odc.stac import load as odc_load

S2_BANDS = ['B02','B03','B04','B08','B05','B06','B07','B8A','B11','B12','B01','B09']
S1_BANDS = ['vv','vh']
STAC = 'https://planetarycomputer.microsoft.com/api/stac/v1'
cat = pystac_client.Client.open(STAC, modifier=pc.sign_inplace)

def _center_crop(a, size):
    H, W = a.shape[-2], a.shape[-1]
    if H < size or W < size:
        a = np.pad(a, [(0,0)]*(a.ndim-2)+[(0,max(0,size-H)),(0,max(0,size-W))], constant_values=np.nan)
        H, W = a.shape[-2], a.shape[-1]
    t, l = (H-size)//2, (W-size)//2
    return a[..., t:t+size, l:l+size]

def extract_patch(lon, lat, year, month, size=32, cloud_lt=40):
    start = f'{year}-{month:02d}-01'
    ed = [31,29 if year%4==0 else 28,31,30,31,30,31,31,30,31,30,31][month-1]
    end = f'{year}-{month:02d}-{ed:02d}'
    half = size*10/2/111320
    bbox = [lon-half*1.4, lat-half*1.4, lon+half*1.4, lat+half*1.4]
    s2it = list(cat.search(collections=['sentinel-2-l2a'], bbox=bbox, datetime=f'{start}/{end}',
                           query={'eo:cloud_cover':{'lt':cloud_lt}}).items())
    if not s2it: return None, None
    ds = odc_load(s2it, bands=S2_BANDS+['SCL'], bbox=bbox, resolution=10, crs='utm', groupby='solar_day', chunks={})
    bad = ds['SCL'].isin([0,1,3,8,9,10,11])
    s2 = np.nan_to_num(_center_crop(ds[S2_BANDS].where(~bad).to_array('band').median('time').values.astype(np.float32), size), nan=0.0)
    s1 = None
    s1it = list(cat.search(collections=['sentinel-1-rtc'], bbox=bbox, datetime=f'{start}/{end}').items())
    if s1it:
        arr = odc_load(s1it, bands=S1_BANDS, groupby='solar_day', chunks={}, like=ds)[S1_BANDS].to_array('band').mean('time').values.astype(np.float32)
        s1 = np.nan_to_num(_center_crop(10*np.log10(np.clip(arr,1e-6,None)), size), nan=-30.0)
    return s2, s1

def extract_set(features, limit=None):
    S2, S1, Y, MO, YR = [], [], [], [], []
    feats = features[:limit] if limit else features
    for i, ft in enumerate(feats):
        lon, lat = ft['geometry']['coordinates'][:2]; p = ft['properties']
        yr, mo = int(p['year']), int(p['month'])
        s2, s1 = extract_patch(lon, lat, yr, mo, size=32)
        if s2 is None: continue
        if s1 is None: s1 = np.zeros((2, s2.shape[1], s2.shape[2]), np.float32) - 30.0
        S2.append(s2); S1.append(s1); Y.append(p['oe_labels']['category']); MO.append(mo); YR.append(yr)
        if (i+1) % 25 == 0: print(f'  {i+1}/{len(feats)}', flush=True)
    return (np.stack(S2), np.stack(S1), np.array(Y), np.array(MO), np.array(YR))

In [ ]:
# Field points — the validation set. Extract ALL 190 (a few minutes on Colab).
field_only = [f for f in oe_feats if f['properties']['src'] == 'field']
f2, f1, fy, fmo, fyr = extract_set(field_only)
print('field patches:', f2.shape, 'labels:', np.bincount(fy))

In [ ]:
# ODC weak labels — the pretraining set. This is large; start with a subset on Colab (e.g. 400/class),
# raise `ODC_LIMIT=None` for the full curriculum. Extraction is the slow part (network-bound).
ODC_LIMIT = 300
odc_only = [f for f in oe_feats if f['properties']['src'] == 'odc']
o2, o1, oy, omo, oyr = extract_set(odc_only, limit=ODC_LIMIT)
have_cache = len(oy) > 0    # gate the fine-tune curriculum on having ODC data
print('odc patches  :', o2.shape, 'labels:', np.bincount(oy))

# (Optional, local olmoearth env only) load prebuilt caches instead of extracting:
#   d = np.load('limpopo/data/patches/limpopo_val.npz', allow_pickle=True)
#   f2,f1,fy,fmo,fyr = [d[k] for k in ('s2','s1','y','month','year')]

## 4. Load OlmoEarth-LARGE

In [ ]:
from olmoearth_pretrain.model_loader import ModelID, load_model_from_id
from olmoearth_pretrain.data.normalize import Normalizer, Strategy
from olmoearth_pretrain.data.constants import Modality
from olmoearth_pretrain.datatypes import MaskedOlmoEarthSample, MaskValue

NORM = Normalizer(Strategy.COMPUTED, std_multiplier=2)
WIN, TRAIN_PS, ONLINE = 32, 4, MaskValue.ONLINE_ENCODER.value
CT = (WIN // TRAIN_PS) // 2   # center token index (train at ps=4, infer at ps=1 for 10 m edges)

encoder = load_model_from_id(ModelID.OLMOEARTH_V1_LARGE).eval().encoder.to(DEV)
print('OlmoEarth-LARGE loaded — patch_size', TRAIN_PS)

### Shared helpers — build a `MaskedOlmoEarthSample` and pool encoder tokens
(identical to `scripts/40_build_three_models.py`; S1 masked when absent).

In [ ]:
def build(s2, s1, mo, yr, idx):
    s2n = np.clip(NORM.normalize(Modality.SENTINEL2_L2A,
           rearrange(np.stack([s2[j] for j in idx]), 'b c h w -> b h w c')[:, :, :, None, :]), 0, 1).astype(np.float32)
    s1n = np.clip(NORM.normalize(Modality.SENTINEL1,
           rearrange(np.stack([s1[j] for j in idx]), 'b c h w -> b h w c')[:, :, :, None, :]), 0, 1).astype(np.float32)
    B = len(idx)
    ts = torch.stack([torch.tensor([15, int(mo[j])-1, int(yr[j])]) for j in idx]).long().to(DEV)[:, None, :]
    return MaskedOlmoEarthSample(
        sentinel2_l2a=torch.from_numpy(s2n).to(DEV),
        sentinel2_l2a_mask=torch.full((B, WIN, WIN, 1, 3), ONLINE, dtype=torch.int32, device=DEV),
        sentinel1=torch.from_numpy(s1n).to(DEV),
        sentinel1_mask=torch.full((B, WIN, WIN, 1, 1), ONLINE, dtype=torch.int32, device=DEV),
        timestamps=ts)

def tokens(enc, sample):
    tam = enc(sample, fast_pass=True, patch_size=TRAIN_PS)['tokens_and_masks']
    g = [getattr(tam, m).float().mean(dim=(3, 4)) for m in ('sentinel2_l2a', 'sentinel1')
         if getattr(tam, m, None) is not None]
    return torch.stack(g, 0).mean(0)   # (B, tokY, tokX, dim) fused across modalities

## 5. Option A — embeddings + linear probe (quick sanity check)

Like the tutorial's Part A: freeze the encoder, take the **center-token** embedding, and fit a
logistic-regression probe. Fast way to validate the signal before fine-tuning.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix

def center_embeddings(s2, s1, mo, yr, bs=8):
    E = []
    for i in range(0, len(s2), bs):
        idx = list(range(i, min(i+bs, len(s2))))
        with torch.no_grad():
            t = tokens(encoder, build(s2, s1, mo, yr, idx))   # (b, tY, tX, d)
        E.append(t[:, CT, CT, :].cpu().numpy())
    return np.concatenate(E)

Xf = center_embeddings(f2, f1, fmo, fyr)
clf = LogisticRegression(max_iter=1000)
pred = cross_val_predict(clf, StandardScaler().fit_transform(Xf), fy, cv=5)
print(f'linear-probe (5-fold): OA={accuracy_score(fy, pred):.3f}  kappa={cohen_kappa_score(fy, pred):.3f}')
print(confusion_matrix(fy, pred))

## 6. Option B — fine-tune with the ODC → field curriculum (best model)

Our production recipe (reproduces `scripts/40_build_three_models.py`, the **OA 0.874** model):
pretrain the encoder+head on the abundant **ODC weak labels**, then fine-tune on the **190 field
points**. A 2-way linear head on the fused center token; class-weighted cross-entropy.

In [ ]:
BS, LR_ENC, LR_HEAD = 8, 2e-5, 1e-3

def train(enc, head, s2, s1, mo, yr, y, epochs):
    cnt = np.bincount(y, minlength=2)
    cw = torch.tensor([len(y)/(2*max(1, cnt[0])), len(y)/(2*max(1, cnt[1]))]).float().to(DEV)
    opt = torch.optim.AdamW([{'params': enc.parameters(), 'lr': LR_ENC},
                             {'params': head.parameters(), 'lr': LR_HEAD}], weight_decay=1e-4)
    lf = nn.CrossEntropyLoss(weight=cw); yt = torch.tensor(y)
    for e in range(epochs):
        enc.train(); perm = np.random.permutation(len(y))
        for i in range(0, len(y), BS):
            b = perm[i:i+BS]; opt.zero_grad()
            logit = head(tokens(enc, build(s2, s1, mo, yr, b))[:, CT, CT, :])
            lf(logit, yt[b.tolist()].to(DEV)).backward(); opt.step()
        print(f'  epoch {e+1}/{epochs}', flush=True)

# Uses the ODC patches (o*) and field patches (f*) extracted in Section 3.
if have_cache:
    np.random.seed(0); torch.manual_seed(0)
    enc = load_model_from_id(ModelID.OLMOEARTH_V1_LARGE).eval().encoder.to(DEV)
    with torch.no_grad():
        dim = tokens(enc, build(f2, f1, fmo, fyr, [0])).shape[-1]
    head = nn.Linear(dim, 2).to(DEV)
    print('ODC-pretrain (5 ep)…');  train(enc, head, o2, o1, omo, oyr, oy, 5)
    print('field fine-tune (8 ep)…'); train(enc, head, f2, f1, fmo, fyr, fy, 8)
    ckpt = WORK / 'notebook_odc_pretrain_large.pt'
    torch.save({'encoder': enc.state_dict(), 'head': head.state_dict(),
                'model_id': 'OLMOEARTH_V1_LARGE', 'train_patch_size': TRAIN_PS, 'infer_patch_size': 1,
                'modalities': ['sentinel2_l2a', 'sentinel1'], 'embed_dim': dim}, ckpt)
    print('saved', ckpt)
else:
    print('No ODC patches extracted — run Section 3 (or lower ODC_LIMIT / check the network).')

## 7. Evaluate — spatial cross-validation

Report **overall accuracy (OA)** and **Cohen's κ** on the 190 field points using **spatial**
GroupKFold (0.5° blocks) so nearby points don't leak between train/val — the honest metric that
produced OA 0.874 / κ 0.748 for the ODC-pretrained model.

In [ ]:
from sklearn.model_selection import GroupKFold

def spatial_groups(features, block=0.5):
    g = []
    for ft in features:
        lon, lat = ft['geometry']['coordinates'][:2]
        g.append((int(lon/block), int(lat/block)))
    uniq = {k: i for i, k in enumerate(sorted(set(g)))}
    return np.array([uniq[k] for k in g])

# groups aligned to the field embeddings Xf/fy computed in Section 5
groups = spatial_groups(field_only[:len(fy)])
gkf = GroupKFold(n_splits=min(5, len(set(groups))))
pred = cross_val_predict(LogisticRegression(max_iter=1000),
                         StandardScaler().fit_transform(Xf), fy, groups=groups, cv=gkf)
print(f'spatial CV (probe): OA={accuracy_score(fy, pred):.3f}  kappa={cohen_kappa_score(fy, pred):.3f}')
print('(the fine-tuned LARGE curriculum reaches OA 0.874 / kappa 0.748 on the full 190)')

### Class definitions

Binary irrigation classes (mirrors the tutorial's class table, adapted to our use case):

| ID | Class | Description |
|----|-------|-------------|
| 0 | Non-irrigated | Rainfed / dry cropland (and buffered non-crop negatives) |
| 1 | Irrigated | Actively irrigated cropland in the dry season |

In [ ]:
import matplotlib.pyplot as plt
CLASS_NAMES = ['Non-irrigated', 'Irrigated']

# Sample S2 RGB patches per class (mirrors the tutorial's 'Sample Images for Each Class').
# S2_BANDS order is [B02,B03,B04,...] so RGB = indices [2,1,0]; /3000 for display.
def rgb(s2):
    return np.clip(np.stack([s2[2], s2[1], s2[0]], -1) / 3000.0, 0, 1)

fig, axes = plt.subplots(2, 4, figsize=(11, 5.5))
for cls in (0, 1):
    ex = np.where(fy == cls)[0][:4]
    for j, ax in enumerate(axes[cls]):
        if j < len(ex): ax.imshow(rgb(f2[ex[j]]))
        ax.set_title(f'{cls}: {CLASS_NAMES[cls]}', fontsize=9); ax.axis('off')
plt.suptitle('Sample Sentinel-2 RGB patches per class'); plt.tight_layout(); plt.show()

### Confusion matrix (row-normalized)

Same style as the tutorial — rows sum to 1, so the diagonal is per-class recall.

In [ ]:
from sklearn.metrics import confusion_matrix

# `pred`/`fy` come from the spatial-CV in the previous cell (honest, leakage-free).
cm = confusion_matrix(fy, pred, labels=[0, 1])
cmn = cm / cm.sum(axis=1, keepdims=True).clip(min=1)

fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(CLASS_NAMES); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True (Ground Truth)')
for i in (0, 1):
    for j in (0, 1):
        ax.text(j, i, f'{cmn[i, j]:.2f}', ha='center', va='center',
                color='white' if cmn[i, j] > 0.5 else 'black', fontsize=12)
fig.colorbar(im, ax=ax, fraction=0.046)
ax.set_title('Confusion matrix (row-normalized, spatial CV)')
plt.tight_layout(); plt.show()
print('counts:\n', cm)

### Per-class precision / recall / F1

The tutorial's `classification_report` — precision, recall and F1 for each class (spatial CV).

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(fy, pred, labels=[0, 1], target_names=CLASS_NAMES, digits=3))

### Segmentation comparison — Embeddings vs Fine-tuned

Like the tutorial's comparison panel. For a few field windows, side by side:
**RGB · Ground-truth center label · Embedding + linear-probe** (frozen encoder, coarse ps=4) **·
Fine-tuned** (10 m, ps=1). Shows why fine-tuning gives cleaner, sharper maps than the frozen probe.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
assert 'head' in globals() and 'Xf' in globals(), 'Run Section 5 (Xf) and Section 6 (enc, head) first.'

# fit the embedding probe on the field center-token embeddings (frozen `encoder` from Section 4)
_scaler = StandardScaler().fit(Xf)
_probe = LogisticRegression(max_iter=1000).fit(_scaler.transform(Xf), fy)
SEG_CMAP = ListedColormap(['#c8a45a', '#1a9850'])   # 0 non-irrigated, 1 irrigated

def _tok_grid(model, s2p, s1p, mo, yr, ps):
    samp = build(np.stack([s2p]), np.stack([s1p]), [mo], [yr], [0])
    with torch.no_grad():
        tam = model(samp, fast_pass=True, patch_size=ps)['tokens_and_masks']
    g = [getattr(tam, m).float().mean(dim=(3,4)) for m in ('sentinel2_l2a','sentinel1') if getattr(tam, m, None) is not None]
    return torch.stack(g, 0).mean(0)[0].cpu().numpy()   # (tY, tX, dim)

picks = []
for c in (1, 0):                       # one irrigated, one non-irrigated example
    w = np.where(fy == c)[0]
    if len(w): picks.append(int(w[0]))

fig, axes = plt.subplots(len(picks), 4, figsize=(12, 3.2*len(picks)))
if len(picks) == 1: axes = axes[None, :]
titles = ['RGB', 'Ground Truth', 'Embedding + LP', 'Fine-tuned']
for r, i in enumerate(picks):
    axes[r,0].imshow(rgb(f2[i])); axes[r,0].set_ylabel(CLASS_NAMES[int(fy[i])], fontsize=9)
    gt = np.full((32,32), np.nan); gt[13:19, 13:19] = fy[i]      # center label
    axes[r,1].imshow(np.ones((32,32,3))); axes[r,1].imshow(gt, cmap=SEG_CMAP, vmin=0, vmax=1)
    fe = _tok_grid(encoder, f2[i], f1[i], fmo[i], fyr[i], 4)     # frozen, coarse (8x8)
    seg_lp = _probe.predict(_scaler.transform(fe.reshape(-1, fe.shape[-1]))).reshape(fe.shape[:2])
    axes[r,2].imshow(seg_lp, cmap=SEG_CMAP, vmin=0, vmax=1, interpolation='nearest')
    ff = _tok_grid(enc, f2[i], f1[i], fmo[i], fyr[i], 1)         # fine-tuned, 10 m (32x32)
    with torch.no_grad():
        seg_ft = torch.softmax(head(torch.from_numpy(ff).to(DEV)), -1).argmax(-1).cpu().numpy()
    axes[r,3].imshow(seg_ft, cmap=SEG_CMAP, vmin=0, vmax=1, interpolation='nearest')
    for k in range(4):
        if r == 0: axes[r,k].set_title(titles[k], fontsize=10)
        axes[r,k].set_xticks([]); axes[r,k].set_yticks([])
plt.suptitle('Segmentation comparison: Embeddings vs Fine-tuned', fontweight='bold')
plt.tight_layout(); plt.show()

## 8. Predict — map irrigation over an AOI at 10 m (runs in Colab)

Uses the fine-tuned `(enc, head)` from Section 6. Fetches S2+S1 for a **small** AOI + month, masks
to WorldCover cropland, tiles it, runs the encoder at **`patch_size=1`** (10 m tokens), classifies
every token, and plots the map + irrigated-area stats. Keep the AOI small (~10×10 km) on a T4.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap; from matplotlib.patches import Patch

AOI = [29.20, -25.00, 29.34, -24.88]   # small box, Middle Olifants (~13x13 km). Keep small on T4.
PMONTH, PYEAR = 9, 2024                 # dry-season month/year to map
assert 'head' in globals() and 'enc' in globals(), 'Run Section 6 first to fine-tune (enc, head).'
enc.eval()

# --- fetch imagery for the whole AOI ---
s2it = list(cat.search(collections=['sentinel-2-l2a'], bbox=AOI,
            datetime=f'{PYEAR}-{PMONTH:02d}-01/{PYEAR}-{PMONTH:02d}-28',
            query={'eo:cloud_cover': {'lt': 40}}).items())
ds = odc_load(s2it, bands=S2_BANDS+['SCL'], bbox=AOI, resolution=10, crs='utm', groupby='solar_day', chunks={})
bad = ds['SCL'].isin([0,1,3,8,9,10,11])
S2 = np.nan_to_num(ds[S2_BANDS].where(~bad).to_array('band').median('time').values.astype(np.float32), nan=0.0)
s1it = list(cat.search(collections=['sentinel-1-rtc'], bbox=AOI,
            datetime=f'{PYEAR}-{PMONTH:02d}-01/{PYEAR}-{PMONTH:02d}-28').items())
if s1it:
    S1 = np.nan_to_num(10*np.log10(np.clip(odc_load(s1it, bands=S1_BANDS, groupby='solar_day', chunks={}, like=ds)[S1_BANDS].to_array('band').mean('time').values.astype(np.float32), 1e-6, None)), nan=-30.0)
else:
    S1 = np.full((2,)+S2.shape[1:], -30.0, np.float32)

# --- cropland mask from ESA WorldCover ---
wc = odc_load(list(cat.search(collections=['esa-worldcover'], bbox=AOI).items()), bands=['map'], chunks={}, like=ds)['map'].isel(time=0).values
H, W = S2.shape[1], S2.shape[2]; cropland = (wc == 40)
nY, nX = H//WIN, W//WIN; Hc, Wc = nY*WIN, nX*WIN; cropland = cropland[:Hc, :Wc]
print(f'AOI {H}x{W}px, cropland px={int(cropland.sum())} — running LARGE at ps=1 (10 m)…')

In [ ]:
# --- tile + dense inference at ps=1 ---
def _build_tiles(tiles):
    B = len(tiles)
    s2n = np.clip(NORM.normalize(Modality.SENTINEL2_L2A, rearrange(np.stack([t[0] for t in tiles]), 'b c h w -> b h w c')[:,:,:,None,:]), 0, 1).astype(np.float32)
    s1n = np.clip(NORM.normalize(Modality.SENTINEL1,    rearrange(np.stack([t[1] for t in tiles]), 'b c h w -> b h w c')[:,:,:,None,:]), 0, 1).astype(np.float32)
    ts = torch.tensor([[15, PMONTH-1, PYEAR]]*B).long().to(DEV)[:, None, :]
    return MaskedOlmoEarthSample(
        sentinel2_l2a=torch.from_numpy(s2n).to(DEV),
        sentinel2_l2a_mask=torch.full((B,WIN,WIN,1,3), ONLINE, dtype=torch.int32, device=DEV),
        sentinel1=torch.from_numpy(s1n).to(DEV),
        sentinel1_mask=torch.full((B,WIN,WIN,1,1), ONLINE, dtype=torch.int32, device=DEV), timestamps=ts)

prob = np.zeros((Hc, Wc), np.float32); buf, pos = [], []
def _flush():
    global buf, pos
    if not buf: return
    with torch.no_grad():
        tam = enc(_build_tiles(buf), fast_pass=True, patch_size=1)['tokens_and_masks']
        g = [getattr(tam, m).float().mean(dim=(3,4)) for m in ('sentinel2_l2a','sentinel1') if getattr(tam, m, None) is not None]
        gl = torch.softmax(head(torch.stack(g, 0).mean(0)), -1)[..., 1].cpu().numpy()  # (B,32,32)
    for (ty,tx), gg in zip(pos, gl): prob[ty*WIN:(ty+1)*WIN, tx*WIN:(tx+1)*WIN] = gg
    buf, pos = [], []
for ty in range(nY):
    for tx in range(nX):
        if cropland[ty*WIN:(ty+1)*WIN, tx*WIN:(tx+1)*WIN].mean() < 0.05: continue
        y0, x0 = ty*WIN, tx*WIN
        buf.append((S2[:, y0:y0+WIN, x0:x0+WIN], S1[:, y0:y0+WIN, x0:x0+WIN])); pos.append((ty, tx))
        if len(buf) >= 10: _flush()
_flush()

irr = (prob >= 0.5) & cropland; n_irr = int(irr.sum()); n_crop = int(cropland.sum())
cls = np.zeros((Hc, Wc), np.uint8); cls[cropland & ~irr] = 1; cls[irr] = 2
pct = 100*n_irr/max(1, n_crop); ha = n_irr*100/1e4
print(f'irrigated: {pct:.1f}% of cropland  (~{ha:,.0f} ha of {n_crop*100/1e4:,.0f} ha cropland)')

cmap = ListedColormap(['#eeeeee', '#c8a45a', '#1a9850'])
fig, ax = plt.subplots(figsize=(8, 7))
ax.imshow(cls, cmap=cmap, vmin=0, vmax=2, extent=[AOI[0],AOI[2],AOI[1],AOI[3]], origin='upper')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title(f'Irrigation (10 m) — {PYEAR}-{PMONTH:02d}  ·  {pct:.0f}% of cropland irrigated')
ax.legend(handles=[Patch(color='#1a9850', label='irrigated'), Patch(color='#c8a45a', label='non-irrigated cropland'), Patch(color='#eeeeee', label='non-cropland')], loc='upper left', fontsize=8)
plt.show()

> **Also available without running anything:** the deployed app maps any AOI interactively —
> https://iwmihq-limpopo-irrigation-mapper.hf.space — or `POST /api/predict` (docs at `/api`, `/docs`).

## 9. Considerations & workshop intake-form answers

Answers framed against the official [training-data-considerations](https://docs.olmoearth.allenai.org/training-data-considerations/).

**Use case:** monthly dry-season (May–Sep) irrigated-area mapping at 10 m, Limpopo River Basin.

**Task type:** per-pixel classification (segmentation), Boolean target `irrigated`.

**Data source:** Sentinel-1 RTC (VV/VH) + Sentinel-2 L2A (12 bands) from Microsoft Planetary Computer.

**Labels:** GeoJSON points — Location + `date` + Boolean `irrigated` (0/1). 190 field-truthed points
(validation) + in-basin ODC-distilled weak labels (pretraining). Cropland mask = smallholder-inclusive
union (WorldCover ∪ Dynamic World ∪ SANLC ∪ WorldCereal).

**Dataset size (docs: more labels help — crop-type 1000→F1 0.79, 3000→0.92).** 190 field points is
thin, so we augment with thousands of ODC weak labels and use a curriculum (pretrain on weak → fine-
tune on field). This mirrors the docs' size guidance for a foundation-model fine-tune.

**Data quality (docs: compare all-data vs higher-confidence).** ODC labels are sampled only where the
product's `map` **and** `filtered` bands agree (high-confidence), with negatives buffered ≥3 px from
any irrigation — a direct implementation of the 'higher-confidence subset' recommendation.

**Class balance & negatives (docs).** ODC sampling draws equal irrigated / non-irrigated per month;
cross-entropy is class-weighted. Negatives (non-irrigated cropland) are explicitly included.

**Spatial representativeness (docs: broad > clustered).** Weak labels are sampled basin-wide on a
~330 m grid; evaluation uses **spatial** GroupKFold (0.5° blocks) so clustered points don't leak.

**What's missing / next steps:** ground truth in Zimbabwe, Botswana, Mozambique to confirm transfer;
full monthly × multi-year series; assess S1-absent months.

**Model fit with OlmoEarth:** irrigation is a fine-grained, sub-field, multi-temporal signal — a good
fit for OlmoEarth's 10 m multi-sensor tokens; we chose OlmoEarth (over Prithvi) specifically to map
small smallholder plots.

## Appendix — rslearn / OlmoEarth Studio config (for the workshop tooling)

If you prefer the tutorial's config-driven path, here is the dataset `config.json` **extended with a
Sentinel-1 layer** and a binary `category` label, plus the fine-tune head sized for **LARGE** (1024-d)
and **3 output channels** (0 non-irrigated, 1 irrigated, 2 = nodata). Build with
`rslearn dataset add_windows --window_size 32 --resolution 10 --utm --start … --end …` over the
dry-season window, then `prepare` / `materialize`, then `rslearn model fit`.

In [ ]:
DATASET_CONFIG = {
  'layers': {
    'label': {'type': 'raster', 'band_sets': [{'bands': ['category'], 'dtype': 'int32'}]},
    'sentinel2': {'type': 'raster',
      'band_sets': [
        {'bands': ['B02','B03','B04','B08'], 'dtype': 'uint16'},
        {'bands': ['B05','B06','B07','B8A','B11','B12'], 'dtype': 'uint16', 'zoom_offset': -1},
        {'bands': ['B01','B09'], 'dtype': 'uint16', 'zoom_offset': -2}],
      'data_source': {'class_path': 'rslearn.data_sources.planetary_computer.Sentinel2',
        'ingest': False, 'init_args': {'harmonize': True, 'sort_by': 'eo:cloud_cover'},
        'query_config': {'space_mode': 'MOSAIC', 'max_matches': 1, 'min_matches': 1,
                         'period_duration': '30d'}}},
    'sentinel1': {'type': 'raster',
      'band_sets': [{'bands': ['vv','vh'], 'dtype': 'float32'}],
      'data_source': {'class_path': 'rslearn.data_sources.planetary_computer.Sentinel1',
        'ingest': False, 'init_args': {'sort_by': 'datetime'},
        'query_config': {'space_mode': 'MOSAIC', 'max_matches': 1, 'min_matches': 0,
                         'period_duration': '30d'}}}
  }
}
print(json.dumps(DATASET_CONFIG, indent=2))

In [ ]:
FINETUNE_HEAD = '''
decoders:
  segment:
    - {class_path: rslearn.models.upsample.Upsample, init_args: {scale_factor: 4}}
    - class_path: rslearn.models.conv.Conv
      init_args: {in_channels: 1024, out_channels: 3, kernel_size: 1,
                  activation: {class_path: torch.nn.Identity}}   # 1024 = LARGE embed dim
    - {class_path: rslearn.train.tasks.segmentation.SegmentationHead}
# SegmentationTask(num_classes=3, zero_is_invalid: false, nodata_value: 2)
# encoder: OlmoEarth(model_id: OLMOEARTH_V1_LARGE, patch_size: 4)
'''
print(FINETUNE_HEAD)